<a href="https://colab.research.google.com/github/NehalShahu/Gen_AI/blob/main/Gen_AI_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate scikit-learn

In [12]:
# ============================================================
# BLOCK 1: LOAD + PREPARE DATASET CORRECTLY
# ============================================================

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

import numpy as np


# ============================================================
# 1. LOAD DATASET DIRECTLY FROM HUGGING FACE
# ============================================================

dataset = load_dataset("SetFit/bbc-news")

print(dataset)


# ============================================================
# 2. CHECK ORIGINAL LABEL MAPPING
# ============================================================

print("\nChecking original labels:")

for i in range(10):

    print(
        "Numeric label:",
        dataset["train"][i]["label"],
        "| Text label:",
        dataset["train"][i]["label_text"]
    )


# ============================================================
# 3. CREATE LABEL MAPPING FROM ACTUAL DATA
# ============================================================

# Get the actual relationship between numeric label
# and label_text from the dataset.

label_mapping = {}

for example in dataset["train"]:

    numeric_label = example["label"]
    text_label = example["label_text"]

    label_mapping[numeric_label] = text_label


print("\nOriginal dataset label mapping:")

for numeric_label, text_label in sorted(
    label_mapping.items()
):

    print(
        numeric_label,
        "->",
        text_label
    )


# ============================================================
# 4. CREATE CLEAN CLASS LIST
# ============================================================

# Sort using the original numerical label IDs.
# This preserves the dataset's actual mapping.

label_names = [
    label_mapping[i]
    for i in sorted(label_mapping.keys())
]


print("\nClasses:")

for i, label in enumerate(label_names):

    print(
        i,
        "->",
        label
    )


# ============================================================
# 5. CREATE LABEL DICTIONARIES
# ============================================================

id2label = {
    i: label.upper()
    for i, label in enumerate(label_names)
}

label2id = {
    label: i
    for i, label in enumerate(label_names)
}


print("\nID to Label:")
print(id2label)


# ============================================================
# 6. CREATE CLEAN 'labels' COLUMN
# ============================================================

def create_labels(example):

    example["labels"] = label2id[
        example["label_text"]
    ]

    return example


dataset = dataset.map(
    create_labels
)


# ============================================================
# 7. LOAD DISTILBERT TOKENIZER
# ============================================================

model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)


# ============================================================
# 8. TOKENIZE TEXT
# ============================================================

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)


# ============================================================
# 9. REMOVE ORIGINAL LABEL COLUMN
# ============================================================

# This is important.
# We want Trainer to use ONLY our clean "labels" column.

tokenized_dataset = tokenized_dataset.remove_columns(
    ["label"]
)


# ============================================================
# 10. CHECK FINAL DATASET
# ============================================================

print("\nFinal dataset:")
print(tokenized_dataset)


print("\nFinal training example:")

print(
    tokenized_dataset["train"][0]
)


# ============================================================
# 11. VERIFY LABELS
# ============================================================

print("\nVerifying first 10 labels:")

for i in range(10):

    print(
        "label_text =",
        dataset["train"][i]["label_text"],
        "| labels =",
        tokenized_dataset["train"][i]["labels"]
    )

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 1225
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 1000
    })
})

Checking original labels:
Numeric label: 2 | Text label: sport
Numeric label: 1 | Text label: business
Numeric label: 3 | Text label: entertainment
Numeric label: 1 | Text label: business
Numeric label: 0 | Text label: tech
Numeric label: 4 | Text label: politics
Numeric label: 3 | Text label: entertainment
Numeric label: 3 | Text label: entertainment
Numeric label: 4 | Text label: politics
Numeric label: 1 | Text label: business

Original dataset label mapping:
0 -> tech
1 -> business
2 -> sport
3 -> entertainment
4 -> politics

Classes:
0 -> tech
1 -> business
2 -> sport
3 -> entertainment
4 -> politics

ID to Label:
{0: 'TECH', 1: 'BUSINESS', 2: 'SPORT', 3: 'ENTERTAINMENT', 4: 'POLITICS'}


Map:   0%|          | 0/1225 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1225 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


Final dataset:
DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1225
    })
    test: Dataset({
        features: ['text', 'label_text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1000
    })
})

Final training example:
{'text': 'wales want rugby league training wales could follow england s lead by training with a rugby league club.  england have already had a three-day session with leeds rhinos  and wales are thought to be interested in a similar clinic with rivals st helens. saints coach ian millward has given his approval  but if it does happen it is unlikely to be this season. saints have a week s training in portugal next week  while wales will play england in the opening six nations match on 5 february.  we have had an approach from wales   confirmed a saints spokesman.  it s in the very early stages but it is something we are giving serious

In [13]:
# ============================================================
# BLOCK 2: MODEL BUILDING + TRAINING
# ============================================================

import numpy as np

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)


# ============================================================
# 1. LOAD A COMPLETELY FRESH PRETRAINED MODEL
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(

    model_name,

    num_labels=len(label_names),

    id2label=id2label,

    label2id=label2id
)


# ============================================================
# 2. DATA COLLATOR
# ============================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


# ============================================================
# 3. METRICS
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# ============================================================
# 4. TRAINING SETTINGS
# ============================================================

training_args = TrainingArguments(

    output_dir="./bbc_distilbert_final",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=4,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="accuracy",

    greater_is_better=True,

    logging_steps=20,

    report_to="none"
)


# ============================================================
# 5. CREATE TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    eval_dataset=tokenized_dataset["test"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)


# ============================================================
# 6. TRAIN
# ============================================================

print("============================================")
print("       STARTING MODEL TRAINING")
print("============================================")

trainer.train()

print("\n============================================")
print("       TRAINING COMPLETED")
print("============================================")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


       STARTING MODEL TRAINING


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.111628,0.156041,0.963000,0.965745,0.963000,0.963155
2,0.032524,0.139705,0.963000,0.963883,0.963000,0.963128
3,0.023112,0.120940,0.972000,0.972029,0.972000,0.971985
4,0.008196,0.120948,0.972000,0.972195,0.972000,0.972007


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


       TRAINING COMPLETED


In [14]:
# ============================================================
# BLOCK 3: EVALUATE TRAINED MODEL
# ============================================================

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)


# ============================================================
# 1. EVALUATE ON TEST DATA
# ============================================================

results = trainer.evaluate()


print("============================================")
print("          TEST RESULTS")
print("============================================")


for key, value in results.items():

    if isinstance(value, float):

        print(
            f"{key}: {value:.4f}"
        )

    else:

        print(
            f"{key}: {value}"
        )


# ============================================================
# 2. GET PREDICTIONS
# ============================================================

prediction_output = trainer.predict(
    tokenized_dataset["test"]
)


predicted_labels = np.argmax(
    prediction_output.predictions,
    axis=-1
)


actual_labels = prediction_output.label_ids


# ============================================================
# 3. ACCURACY
# ============================================================

accuracy = accuracy_score(
    actual_labels,
    predicted_labels
)


print("\nTest Accuracy:")
print(
    f"{accuracy * 100:.2f}%"
)


# ============================================================
# 4. CLASSIFICATION REPORT
# ============================================================

print("\n============================================")
print("       CLASSIFICATION REPORT")
print("============================================")


print(
    classification_report(
        actual_labels,
        predicted_labels,

        labels=list(
            range(len(label_names))
        ),

        target_names=[
            id2label[i]
            for i in range(len(label_names))
        ],

        zero_division=0
    )
)


# ============================================================
# 5. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    actual_labels,
    predicted_labels,

    labels=list(
        range(len(label_names))
    )
)


print("\n============================================")
print("          CONFUSION MATRIX")
print("============================================")

print(cm)


# ============================================================
# 6. PRINT MATRIX WITH CLASS NAMES
# ============================================================

print("\nRows = Actual")
print("Columns = Predicted\n")

print(
    "             ",
    " ".join(
        f"{id2label[i]:>15}"
        for i in range(len(label_names))
    )
)

for i, row in enumerate(cm):

    print(
        f"{id2label[i]:>12}",
        " ".join(
            f"{value:15}"
            for value in row
        )
    )

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.008196,0.120940,4,0.972000,0.972029,0.972000,0.971985


          TEST RESULTS
eval_loss: 0.1209
eval_accuracy: 0.9720
eval_precision: 0.9720
eval_recall: 0.9720
eval_f1: 0.9720



Test Accuracy:
97.20%

       CLASSIFICATION REPORT
               precision    recall  f1-score   support

         TECH       0.95      0.97      0.96       189
     BUSINESS       0.96      0.95      0.95       224
        SPORT       1.00      1.00      1.00       236
ENTERTAINMENT       0.98      0.98      0.98       176
     POLITICS       0.97      0.97      0.97       175

     accuracy                           0.97      1000
    macro avg       0.97      0.97      0.97      1000
 weighted avg       0.97      0.97      0.97      1000


          CONFUSION MATRIX
[[183   2   1   3   0]
 [  7 212   0   0   5]
 [  0   1 235   0   0]
 [  2   1   0 172   1]
 [  0   4   0   1 170]]

Rows = Actual
Columns = Predicted

                         TECH        BUSINESS           SPORT   ENTERTAINMENT        POLITICS
        TECH             183               2               1               3               0
    BUSINESS               7             212               0               0      

In [16]:
# ============================================================
# BLOCK 4: CUSTOM USER INPUT + ALL CLASS PROBABILITIES
# ============================================================

import torch


# ============================================================
# 1. PREDICTION FUNCTION
# ============================================================

def classify_news(text):

    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )


    # Move to model device
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }


    # Prediction
    model.eval()

    with torch.no_grad():

        outputs = model(**inputs)


    # Convert logits to probabilities
    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]


    # Sort classes by probability
    sorted_results = torch.argsort(
        probabilities,
        descending=True
    )


    return probabilities, sorted_results


# ============================================================
# 2. ASK USER FOR TEXT
# ============================================================

print("============================================")
print("          BBC NEWS CLASSIFIER")
print("============================================")

print("\nAvailable categories:")

for i in range(len(label_names)):

    print(
        f"{i}: {id2label[i]}"
    )


print("\nEnter your news article:")
print()


user_text = input("> ")


# ============================================================
# 3. CHECK INPUT
# ============================================================

if user_text.strip() == "":

    print("\nNo text was entered.")


else:

    probabilities, sorted_results = (
        classify_news(user_text)
    )


    # --------------------------------------------------------
    # TOP PREDICTION
    # --------------------------------------------------------

    predicted_id = sorted_results[0].item()

    predicted_label = id2label[
        predicted_id
    ]

    confidence = probabilities[
        predicted_id
    ].item()


    # --------------------------------------------------------
    # DISPLAY TOP RESULT
    # --------------------------------------------------------

    print("\n============================================")
    print("                RESULT")
    print("============================================")

    print("\nPredicted Category:")
    print(
        predicted_label
    )

    print("\nConfidence:")
    print(
        f"{confidence * 100:.2f}%"
    )


    # --------------------------------------------------------
    # DISPLAY ALL PROBABILITIES
    # --------------------------------------------------------

    print("\n============================================")
    print("       CLASS PROBABILITIES")
    print("============================================")


    for class_id in sorted_results:

        class_id = class_id.item()

        probability = probabilities[
            class_id
        ].item()


        print(
            f"{id2label[class_id]:15s}"
            f" : {probability * 100:6.2f}%"
        )


    print("\n============================================")

          BBC NEWS CLASSIFIER

Available categories:
0: TECH
1: BUSINESS
2: SPORT
3: ENTERTAINMENT
4: POLITICS

Enter your news article:

> White House to roll out the red carpet for historic Xi Jinping visit

                RESULT

Predicted Category:
POLITICS

Confidence:
52.48%

       CLASS PROBABILITIES
POLITICS        :  52.48%
BUSINESS        :  19.04%
ENTERTAINMENT   :  13.19%
SPORT           :   9.48%
TECH            :   5.81%

